# 02 KR2036 網路分析（kwak_full 情境）

改寫自上游 `notebooks/network_analysis.ipynb`，保留原本的章節順序，全部改跑
**kwak_full**（10 節點 / 3H）情境。

**與原版的主要差異**

| 項目 | 上游原版 | 韓國版 |
|---|---|---|
| 網路檔定位 | wildcard 搜尋 `elec_s{simpl}_{clusters}_ec_l{ll}_{opts}.nc` | 直接讀 `_kr_common.SCENARIOS` 的絕對路徑，不猜檔名 |
| 地圖投影 | `ccrs.EqualEarth()` + 自動 bounds | `ccrs.PlateCarree()` + 固定 `KR_EXTENT`（否則離岸風節點會把畫面拉到海上）|
| load shedding | 只在部分圖排除 | `load_network()` 一律先移除，所有容量／發電／圓餅都不含虛擬機組 |
| 風光潛力圖 | 只有 onwind / solar | 另加 **offwind-ac / offwind-dc**（用離岸 voronoi 區域）|
| 國家過濾 | `filter(regex="NG *")` 字串比對 | 不過濾（全網只有 KR）|

## 0. 環境設定與載入網路

In [ ]:
import sys
import warnings
from pathlib import Path

_here = Path.cwd()
for cand in (_here, _here / "notebooks_kr", _here.parent):
    if (cand / "_kr_common.py").exists():
        sys.path.insert(0, str(cand))
        break

import _kr_common as K

K.setup_matplotlib()

import cartopy
import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import xarray as xr
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
from pypsa.plot import add_legend_circles, add_legend_lines, add_legend_patches

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

SCEN = "kwak_full"
info = K.SCENARIOS[SCEN]
print("情境：", info["long_label"])
print("網路檔：", info["path"])

### 0.1 load shedding 虛擬機組

`config.yaml` 開了 `load_shedding`，網路裡有 10 台虛擬機組（每個節點一台）。
容量高達 137 GW，如果不排除，所有容量圖與圓餅都會被它灌爆。
下面先量出它的規模再移除，之後所有分析都不含這些機組。

In [ ]:
n_raw = K.load_network(SCEN, drop_load_shedding=False)
shed = n_raw.generators[n_raw.generators.carrier.str.contains("load", case=False)]
shed_mwh = (
    n_raw.generators_t.p[shed.index]
    .mul(n_raw.snapshot_weightings.generators, axis=0)
    .sum()
    .sum()
)
print(f"load shedding 機組：{len(shed)} 台，p_nom 合計 {shed.p_nom.sum() / 1e3:,.1f} GW")
print(f"全年實際動用：{shed_mwh:,.2f} MWh（= {shed_mwh / 1e6:.9f} TWh）")
print("→ 容量很大但幾乎沒用到，代表系統充裕度足夠；以下分析一律排除。")

n = K.load_network(SCEN)  # 預設已排除 load shedding

## 1. 資料載入檢查

In [ ]:
comp_rows = []
for comp in n.iterate_components(list(n.components.keys())[2:]):
    if len(comp.df):
        comp_rows.append({"元件": comp.name, "筆數": len(comp.df)})
components = pd.DataFrame(comp_rows).set_index("元件")

print(f"時間解析度：{len(n.snapshots)} 個 snapshot（權重 {n.snapshot_weightings.generators.unique()} 小時）")
print(f"起訖：{n.snapshots[0]} → {n.snapshots[-1]}")
print(f"DC link 數：{len(n.links)}（10 節點下濟州併入本土叢集，因此為空）")
components

## 2. 區域圖（voronoi 叢集區域）

10 節點的陸域 voronoi 區域，以及離岸風可用的離岸區域。
**濟州在 10 節點下併入本土叢集**，要到 30 節點才會切成獨立節點。

In [ ]:
regions_on = K.load_regions(10, "onshore")
regions_off = K.load_regions(10, "offshore")

fig, ax = plt.subplots(figsize=(8, 9), subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_extent(K.KR_EXTENT, crs=ccrs.PlateCarree())

regions_off.plot(
    ax=ax, facecolor="#cfe3f2", edgecolor="#7fa8c9", linewidth=0.5,
    alpha=0.6, transform=ccrs.PlateCarree(), zorder=1,
)
regions_on.plot(
    ax=ax, facecolor="none", edgecolor="#333333", linewidth=1.0,
    transform=ccrs.PlateCarree(), zorder=2,
)
regions_on.plot(
    ax=ax, column=regions_on.index.to_series(), cmap="Pastel2",
    alpha=0.75, transform=ccrs.PlateCarree(), zorder=1,
)

for name, row in regions_on.iterrows():
    ax.annotate(
        name, xy=(row.x, row.y), ha="center", va="center", fontsize=9,
        fontweight="bold", transform=ccrs.PlateCarree(), zorder=4,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75),
    )

ax.set_title(f"10 節點叢集區域（陸域 {len(regions_on)} 區 / 離岸 {len(regions_off)} 區）", fontsize=13, pad=10)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.4)
gl.top_labels = False
gl.right_labels = False

K.savefig("02_fig1_regions_10n")
plt.show()

area = regions_on.to_crs(K.KR_EQUAL_AREA_CRS).geometry.area * 1e-6
print(f"陸域區域總面積：{area.sum():,.0f} km²（南韓國土約 100,200 km²）")

## 3. 容量地圖

### 3.1 繪圖設定

`bus_size_factor` 與 `linewidth_factor` 沿用 `scripts/plot_network.py` 已修好的韓國尺度
（2e6 / 1e4），配色取 `config.yaml` 的 `plotting.tech_colors`，與
`results/compare/kwak_full_vs_paper/` 既有圖表同源。

In [ ]:
BUS_SIZE_FACTOR = 2e6
LINEWIDTH_FACTOR = 1e4


def capacity_map(n, attr, title, fname):
    gen = n.generators.groupby(["bus", "carrier"])[attr].sum()
    sto_attr = attr if attr in n.storage_units.columns else "p_nom"
    sto = n.storage_units.groupby(["bus", "carrier"])[sto_attr].sum()
    sizes = pd.concat([gen, sto])
    sizes = sizes[sizes > 0]

    carriers = sizes.index.get_level_values(1).unique()
    colors = {c: K.carrier_color(c) for c in carriers}

    line_attr = "s_nom_opt" if attr == "p_nom_opt" else "s_nom"

    fig, ax = plt.subplots(figsize=(8, 9), subplot_kw={"projection": ccrs.PlateCarree()})
    n.plot(
        ax=ax,
        bus_sizes=sizes / BUS_SIZE_FACTOR,
        bus_colors=colors,
        bus_alpha=0.85,
        line_widths=n.lines[line_attr] / LINEWIDTH_FACTOR,
        line_colors="#70af1d",
        link_widths=0,  # 10 節點沒有 DC link，n.links 為空
        geomap=True,
        color_geomap={"ocean": "white", "land": "whitesmoke"},
        boundaries=K.KR_EXTENT,
    )

    add_legend_circles(
        ax,
        [s / BUS_SIZE_FACTOR for s in (5e3, 20e3)],
        ["5 GW", "20 GW"],
        legend_kw=dict(loc="upper left", bbox_to_anchor=(0.0, 1.0), frameon=True, labelspacing=1.8),
    )
    add_legend_lines(
        ax,
        [s / LINEWIDTH_FACTOR for s in (5e3, 20e3)],
        ["5 GW", "20 GW"],
        legend_kw=dict(loc="upper left", bbox_to_anchor=(0.0, 0.78), frameon=True),
    )
    order = sizes.groupby(level=1).sum().sort_values(ascending=False).index
    add_legend_patches(
        ax,
        [K.carrier_color(x) for x in order],
        [f"{K.carrier_label(x)}  {sizes.groupby(level=1).sum()[x] / 1e3:,.1f} GW" for x in order],
        legend_kw=dict(loc="lower left", bbox_to_anchor=(1.0, 0.0), frameon=False, fontsize=9),
    )

    ax.set_title(title, fontsize=13, pad=10)
    K.savefig(fname)
    plt.show()
    return sizes

### 3.2 現有容量（`p_nom`，最佳化前）

In [ ]:
sizes_installed = capacity_map(
    n, "p_nom",
    f"KR2036 {info['label']}：現有裝置容量（p_nom）",
    "02_fig2_capacity_map_installed",
)
print(f"現有容量合計：{sizes_installed.sum() / 1e3:,.1f} GW")

### 3.3 最佳化後容量（`p_nom_opt`）

In [ ]:
sizes_optimal = capacity_map(
    n, "p_nom_opt",
    f"KR2036 {info['label']}：最佳化後裝置容量（p_nom_opt）",
    "02_fig3_capacity_map_optimal",
)
print(f"最佳化後容量合計：{sizes_optimal.sum() / 1e3:,.1f} GW")

## 4. 容量圓餅：最佳化前後對照

左圖是既有機組（`p_nom`），右圖是最佳化後（`p_nom_opt`）。

In [ ]:
def capacity_by_carrier(n, attr):
    sto_attr = attr if attr in n.storage_units.columns else "p_nom"
    s = pd.concat([
        n.generators.groupby("carrier")[attr].sum(),
        n.storage_units.groupby("carrier")[sto_attr].sum(),
    ]).div(1e3)
    return s[s > 0].sort_values(ascending=False)


cap_installed = capacity_by_carrier(n, "p_nom")
cap_optimal = capacity_by_carrier(n, "p_nom_opt")

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, data, sub in zip(axes, [cap_installed, cap_optimal], ["現有（p_nom）", "最佳化後（p_nom_opt）"]):
    colors = [K.carrier_color(x) for x in data.index]
    wedges, _ = ax.pie(data, colors=colors, startangle=90, counterclock=False)
    ax.set_title(f"{sub}　合計 {data.sum():,.1f} GW", fontsize=12)
    ax.legend(
        wedges,
        [f"{K.carrier_label(i)}  {v:,.1f} GW（{100 * v / data.sum():.1f}%）" for i, v in data.items()],
        loc="upper center", bbox_to_anchor=(0.5, -0.02), fontsize=9, frameon=False,
    )
    ax.set_aspect("equal")
    ax.grid(False)

fig.suptitle(f"KR2036 {info['label']}：裝置容量結構（已排除 load shedding）", fontsize=14)
K.savefig("02_fig4_capacity_pie")
plt.show()

pd.DataFrame({"現有_GW": cap_installed, "最佳化後_GW": cap_optimal}).fillna(0).round(2)

## 5. 容量擴充量長條圖

`n.statistics.optimal_capacity()` − `n.statistics.installed_capacity()`。

注意 `n.statistics` 回傳的 carrier 是 **nice_name**（如 `Combined-Cycle Gas`），
`_kr_common.carrier_key()` 會反查回原始 carrier 以對上配色與中文標籤。
離岸風（DC）沒有既有機組，相減時要 `fill_value=0`，否則會變成 NaN。

In [ ]:
optimal = n.statistics.optimal_capacity(comps=["Generator"]).droplevel(0).div(1e3)
installed = n.statistics.installed_capacity(comps=["Generator"]).droplevel(0).div(1e3)
expansion = optimal.sub(installed, fill_value=0).sort_values(ascending=False)
expansion = expansion[expansion.abs() > 1e-6]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(
    [K.carrier_label(i) for i in expansion.index],
    expansion.values,
    color=[K.carrier_color(i) for i in expansion.index],
    edgecolor="white", linewidth=0.8,
)
ax.axhline(0, color="#444", linewidth=0.8)
ax.set_title(f"KR2036 {info['label']}：發電容量擴充量（最佳化後 − 既有）", fontsize=13, pad=10)
ax.set_ylabel("容量擴充 [GW]")
ax.tick_params(axis="x", rotation=30)
for i, v in enumerate(expansion.values):
    ax.text(i, v + (0.6 if v >= 0 else -1.2), f"{v:,.1f}", ha="center", fontsize=9)

K.savefig("02_fig5_capacity_expansion")
plt.show()

pd.DataFrame({
    "既有_GW": installed.reindex(expansion.index).fillna(0),
    "最佳化後_GW": optimal.reindex(expansion.index).fillna(0),
    "擴充_GW": expansion,
}).round(2)

## 6. 能量平衡（`n.statistics.energy_balance()`）

本機 pypsa 0.30.3 的 `energy_balance()` 回傳三層 index
（`component` / `carrier` / `bus_carrier`），carrier 一樣是 nice_name。
正值為供給、負值為需求（`-` 是負載，儲能充電也是負的）。

In [ ]:
eb = n.statistics.energy_balance()
eb_carrier = eb.groupby("carrier").sum().div(1e6)  # MWh → TWh
eb_carrier = eb_carrier[eb_carrier.abs() > 1e-9].sort_values(ascending=False)

supply = eb_carrier[eb_carrier > 0]
demand = eb_carrier[eb_carrier < 0]

fig, ax = plt.subplots(figsize=(11, 4.2))
left = 0.0
for name, val in supply.items():
    ax.barh(0, val, left=left, color=K.carrier_color(name), edgecolor="white", linewidth=0.6, height=0.5)
    left += val
left = 0.0
for name, val in demand.items():
    ax.barh(-0.75, val, left=left, color=K.carrier_color(name), edgecolor="white", linewidth=0.6, height=0.5)
    left += val

ax.axvline(0, color="#444", linewidth=0.8)
ax.set_yticks([0, -0.75])
ax.set_yticklabels(["供給", "需求"])
ax.set_xlabel("TWh")
ax.set_title(f"KR2036 {info['label']}：年度能量平衡", fontsize=13, pad=10)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f"{int(x):,}"))
ax.grid(axis="x", linestyle=":", alpha=0.6)

handles = [
    Patch(
        facecolor=K.carrier_color(i),
        label=f"{'負載' if i == '-' else K.carrier_label(i)}  {v:,.1f} TWh",
    )
    for i, v in eb_carrier.items()
]
ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9, frameon=False)

K.savefig("02_fig6_energy_balance")
plt.show()

### 6.1 能量平衡表（TWh）

In [ ]:
eb_table = pd.DataFrame({"TWh": eb_carrier})
eb_table.index = [K.carrier_label(i) if i != "-" else "負載" for i in eb_table.index]
gen_total = supply.sum()
eb_table["佔總發電%"] = (eb_table["TWh"] / gen_total * 100).where(eb_table["TWh"] > 0)

K.FIG_DIR.mkdir(parents=True, exist_ok=True)
eb_table.round(3).to_csv(K.FIG_DIR / "02_table_energy_balance_TWh.csv", encoding="utf-8-sig")
print("已存檔：", K.FIG_DIR / "02_table_energy_balance_TWh.csv")
print(f"總發電 {gen_total:,.1f} TWh　總負載 {-demand.get('-', 0):,.1f} TWh")
eb_table.round(2)

## 7. 儲能現況：電池與抽蓄幾乎不起作用

這一節是為了避免誤讀前面幾張圖——電池與抽蓄在圖上小到看不見，是**真的幾乎為零**，
不是繪圖漏畫。下面直接把實際數字算出來。

In [ ]:
su = n.storage_units
w = n.snapshot_weightings.stores
rows = []
for carrier in ["battery", "PHS", "hydro"]:
    idx = su.index[su.carrier == carrier]
    if not len(idx):
        continue
    pc = n.storage_units_t.p[idx]
    rows.append({
        "載體": K.carrier_label(carrier),
        "p_nom_opt_MW": su.loc[idx, "p_nom_opt"].sum(),
        "max_hours": su.loc[idx, "max_hours"].unique()[0],
        "放電_GWh": pc.clip(lower=0).mul(w, axis=0).sum().sum() / 1e3,
        "充電_GWh": -pc.clip(upper=0).mul(w, axis=0).sum().sum() / 1e3,
    })
storage = pd.DataFrame(rows).set_index("載體")
storage["淨_GWh"] = storage["放電_GWh"] - storage["充電_GWh"]
storage.round(2)

### 7.1 讀圖注意事項

- **電池**：最佳化後只有 **2.5 MW**（基準情境 2.9 MW），對 260 GW 級的系統而言是
  捨入誤差等級。在容量地圖與圓餅上等同看不見，**不是繪圖漏畫**。
- **抽蓄（PHS）**：帳面 5,308 MW，但 `max_hours = 0`——這是既有 bug
  （`PHS_max_hours` 設定未生效），儲能容量 = `p_nom × max_hours` = 0。
  結果是抽蓄 **全年放電 0 GWh、充電 99.6 GWh**，等於變成一個純損失的寄生負載，
  完全沒有發揮儲能功能。相對 707 TWh 的年需求只有 0.014%，不影響總體結論，
  但**任何關於儲能或調度彈性的討論都不能引用這個抽蓄數字**。
- 這兩點在修好 `PHS_max_hours` 之前都不要當成「模型認為韓國不需要儲能」的證據。

## 8. 風光潛力密度圖（voronoi）

原版只有 onwind / solar，這裡另加 **offwind-ac / offwind-dc**（用離岸 voronoi 區域）。
密度 = 該區域可裝設上限 `p_nom_max` ÷ 區域面積，面積用 **EPSG:5179** 計算
（等面積需求，不能用 PlateCarree 的度數面積）。

In [ ]:
def plot_potential(carrier, kind, cmap, title, fname):
    regions = K.load_regions(10, kind).copy()
    area_km2 = regions.to_crs(K.KR_EQUAL_AREA_CRS).geometry.area * 1e-6

    g = n.generators[n.generators.carrier == carrier]
    p_max = g.groupby("bus").p_nom_max.sum().reindex(regions.index)
    regions["density"] = (p_max / area_km2).fillna(0)

    fig, ax = plt.subplots(figsize=(7.5, 8.5), subplot_kw={"projection": ccrs.PlateCarree()})
    ax.set_extent(K.KR_EXTENT, crs=ccrs.PlateCarree())
    regions.plot(
        ax=ax, column="density", cmap=cmap, linewidth=0.4, edgecolor="k",
        vmin=0, vmax=float(regions["density"].max()), legend=True,
        legend_kwds={"label": "潛力密度 [MW/km²]", "shrink": 0.7},
        transform=ccrs.PlateCarree(),
    )
    ax.coastlines(resolution="50m", linewidth=0.5)
    ax.add_feature(cartopy.feature.BORDERS.with_scale("50m"), linewidth=0.4)
    ax.set_title(title, fontsize=12, pad=10)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.4)
    gl.top_labels = False
    gl.right_labels = False

    K.savefig(fname)
    plt.show()
    return float(p_max.sum()), float((p_max / area_km2).max())


specs = [
    ("onwind", "onshore", "Blues", "陸域風電潛力密度", "02_fig7a_potential_onwind"),
    ("solar", "onshore", "OrRd", "太陽光電潛力密度", "02_fig7b_potential_solar"),
    ("offwind-ac", "offshore", "GnBu", "離岸風電（AC）潛力密度", "02_fig7c_potential_offwind_ac"),
    ("offwind-dc", "offshore", "PuBu", "離岸風電（DC）潛力密度", "02_fig7d_potential_offwind_dc"),
]

pot_rows = []
for carrier, kind, cmap, title, fname in specs:
    total, peak = plot_potential(carrier, kind, cmap, f"KR2036：{title}", fname)
    pot_rows.append({
        "載體": K.carrier_label(carrier),
        "區域": "陸域" if kind == "onshore" else "離岸",
        "潛力上限_GW": total / 1e3,
        "最高密度_MW/km2": peak,
    })

### 8.1 潛力上限彙整

In [ ]:
potential = pd.DataFrame(pot_rows).set_index("載體")
optimal_by_carrier = n.generators.groupby("carrier").p_nom_opt.sum().div(1e3)
potential["最佳化採用_GW"] = [
    optimal_by_carrier.get(c, np.nan) for c in ["onwind", "solar", "offwind-ac", "offwind-dc"]
]
potential["採用率%"] = potential["最佳化採用_GW"] / potential["潛力上限_GW"] * 100

potential.round(2).to_csv(K.FIG_DIR / "02_table_re_potential.csv", encoding="utf-8-sig")
print("已存檔：", K.FIG_DIR / "02_table_re_potential.csv")
potential.round(2)

### 8.2 潛力圖判讀

採用率反映的是「模型把該區域的可用面積用掉多少」。要注意這些 `p_nom_max` 是
`build_renewable_profiles` 依土地／海域可用性算出的技術潛力上限，
**已受 `config.yaml` 的 `agg_p_nom_limits`（第 10 次電力供需基本計畫 2036 目標）約束**，
因此採用率高不代表「地不夠用」，而是政策目標容量先綁住了裝置量。